In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_log = pd.read_csv("log.csv")
df_trace = pd.read_csv("trace.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (per rule 10)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# 1) metric.csv: unique (cmdb_id, kpi_name) with counts, sorted desc
summary_metric = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
    .head(50)
)

# 2) log.csv: unique (cmdb_id, log_name) with counts, sorted desc
summary_log = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
    .head(50)
)

# 3) trace.csv: unique (cmdb_id, trace_name) with counts, sorted desc
summary_trace = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
    .head(50)
)

# 4) error_logs.csv: unique cmdb_id with count of error log rows
summary_error = (
    df_error
    .groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='error_count')
    .sort_values('error_count', ascending=False)
    .reset_index(drop=True)
    .head(50)
)

# Display the four compact summaries (as required, use variables rather than print)
summary_metric, summary_log, summary_trace, summary_error

```
Out[1]:
```
summary_text = (
    "Summary of telemetry files (compact):\n"
    "- metric.csv: Top pairs show the 'carts' service leading with multiple KPIs (cpu, latency-50, latency-90, mem, socket), "
    "each with 25 data points. Many other services also appear with counts up to 25, indicating regular per-minute metric coverage.\n"
    "- log.csv: The highest log counts are for 'catalogue' (log.error_count and log.row_count, 25 points each), with 'orders' and 'front-end' "
    "also showing high counts. Some services/databases (e.g., session-db, orders-db) have far fewer log rows.\n"
    "- trace.csv: No trace telemetry rows were present (empty).\n"
    "- error_logs.csv: No error log rows were present (empty).\n"
    "Interpretation: metric and log telemetry are available and show strong coverage for core services (notably carts and catalogue). "
    "Trace and error-log data are missing in these files."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count
0          carts         cpu     25
1          carts  latency-50     25
2          carts  latency-90     25
3          carts         mem     25
4          carts      socket     25
..           ...         ...    ...
45  queue-master      diskio     25
46  queue-master         mem     25
47  queue-master      socket     25
48      rabbitmq         cpu     25
49      rabbitmq      diskio     25

[50 rows x 3 columns],        cmdb_id         log_name  count
0    catalogue  log.error_count     25
1    catalogue    log.row_count     25
2       orders    log.row_count     25
3       orders  log.error_count     25
4    front-end    log.row_count     25
..         ...              ...    ...
17       carts    log.row_count     20
18  session-db    log.row_count      5
19  session-db  log.error_count      5
20   orders-db  log.error_count      1
21   orders-db    log.row_count      1

[22 rows x 3 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, count]
Index: [], Empty DataFrame
Columns: [cmdb_id, error_count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing dataframes if present, otherwise load
try:
    df_metric
    df_log
    df_trace
    df_error
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_log = pd.read_csv("log.csv")
    df_trace = pd.read_csv("trace.csv")
    df_error = pd.read_csv("error_logs.csv")

# Ensure timestamp columns are parsed to UTC datetimes (rule 10)
for df in [df_metric, df_log, df_trace, df_error]:
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Components and incident window (UTC)
components = ["carts", "catalogue", "orders", "payment", "user"]
window_start = pd.to_datetime("2024-01-18 16:27:59", utc=True)
window_end = pd.to_datetime("2024-01-18 16:57:59", utc=True)

# ------------------------
# 1) Metric preprocessing and thresholding
# ------------------------
# Compute global P95 per (cmdb_id, kpi_name) for the selected components (using full series)
metric_sel = df_metric[df_metric['cmdb_id'].isin(components)].copy()
if not metric_sel.empty:
    p95_metric = (
        metric_sel
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index(name='p95_threshold')
    )
else:
    p95_metric = pd.DataFrame(columns=['cmdb_id','kpi_name','p95_threshold'])

# Filter metric records to incident window
metric_window = metric_sel[(metric_sel['timestamp'] >= window_start) & (metric_sel['timestamp'] <= window_end)].copy()

# Merge p95 into window rows to mark anomalies
if not p95_metric.empty and not metric_window.empty:
    metric_window = metric_window.merge(p95_metric, on=['cmdb_id','kpi_name'], how='left')
    metric_window['is_anomaly'] = metric_window['value'] >= metric_window['p95_threshold']
else:
    metric_window['is_anomaly'] = pd.Series(dtype=bool)

# Aggregates in window per pair
agg_window_metric = (
    metric_window
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        total_points_in_window=('value','size'),
        anomaly_count_in_window=('is_anomaly','sum'),
        earliest_anomaly_timestamp=('timestamp', lambda s: s[metric_window.loc[s.index,'is_anomaly']].min() if len(s)>0 else pd.NaT)
    )
    .reset_index()
)

# Combine with p95 to include pairs that have p95 but zero points in window
metric_summary = p95_metric.merge(agg_window_metric, on=['cmdb_id','kpi_name'], how='left')
# fill NaNs for pairs with no points in window
metric_summary['total_points_in_window'] = metric_summary['total_points_in_window'].fillna(0).astype(int)
metric_summary['anomaly_count_in_window'] = metric_summary['anomaly_count_in_window'].fillna(0).astype(int)
# convert earliest timestamp to ISO string (UTC) or None
metric_summary['earliest_anomaly_timestamp'] = metric_summary['earliest_anomaly_timestamp'].apply(lambda x: pd.NaT if pd.isna(x) else x)
metric_summary['earliest_anomaly_timestamp'] = metric_summary['earliest_anomaly_timestamp'].apply(lambda t: None if pd.isna(t) else t.isoformat())

# Sort by anomaly_count desc, then total_points desc, limit 50
metric_summary = metric_summary.sort_values(['anomaly_count_in_window','total_points_in_window'], ascending=[False,False]).reset_index(drop=True).head(50)

# If there were no p95 pairs (empty), create empty columns for consistency
if metric_summary.empty:
    metric_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','p95_threshold','total_points_in_window','anomaly_count_in_window','earliest_anomaly_timestamp'])

# ------------------------
# 2) Log preprocessing and thresholding
# ------------------------
log_sel = df_log[df_log['cmdb_id'].isin(components)].copy()
if not log_sel.empty:
    p95_log = (
        log_sel
        .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index(name='p95_threshold')
    )
else:
    p95_log = pd.DataFrame(columns=['cmdb_id','log_name','p95_threshold'])

# Filter to window
log_window = log_sel[(log_sel['timestamp'] >= window_start) & (log_sel['timestamp'] <= window_end)].copy()

# Merge p95 into window rows and mark anomalies
if not p95_log.empty and not log_window.empty:
    log_window = log_window.merge(p95_log, on=['cmdb_id','log_name'], how='left')
    log_window['is_anomaly'] = log_window['value'] >= log_window['p95_threshold']
else:
    log_window['is_anomaly'] = pd.Series(dtype=bool)

# Aggregates in window per pair
agg_window_log = (
    log_window
    .groupby(['cmdb_id','log_name'], dropna=False)
    .agg(
        total_points_in_window=('value','size'),
        anomaly_count_in_window=('is_anomaly','sum'),
        earliest_anomaly_timestamp=('timestamp', lambda s: s[log_window.loc[s.index,'is_anomaly']].min() if len(s)>0 else pd.NaT)
    )
    .reset_index()
)

# Combine with p95 to include pairs that have p95 but zero points in window
log_summary = p95_log.merge(agg_window_log, on=['cmdb_id','log_name'], how='left')
log_summary['total_points_in_window'] = log_summary['total_points_in_window'].fillna(0).astype(int)
log_summary['anomaly_count_in_window'] = log_summary['anomaly_count_in_window'].fillna(0).astype(int)
log_summary['earliest_anomaly_timestamp'] = log_summary['earliest_anomaly_timestamp'].apply(lambda x: pd.NaT if pd.isna(x) else x)
log_summary['earliest_anomaly_timestamp'] = log_summary['earliest_anomaly_timestamp'].apply(lambda t: None if pd.isna(t) else t.isoformat())

log_summary = log_summary.sort_values(['anomaly_count_in_window','total_points_in_window'], ascending=[False,False]).reset_index(drop=True).head(50)

if log_summary.empty:
    log_summary = pd.DataFrame(columns=['cmdb_id','log_name','p95_threshold','total_points_in_window','anomaly_count_in_window','earliest_anomaly_timestamp'])

# ------------------------
# 3) Trace and error_logs presence for these components
# ------------------------
trace_counts = df_trace[df_trace['cmdb_id'].isin(components)].groupby('cmdb_id', dropna=False).size().reset_index(name='count')
error_counts = df_error[df_error['cmdb_id'].isin(components)].groupby('cmdb_id', dropna=False).size().reset_index(name='count')

# Summary rows: overall counts and yes/no
trace_overall_count = trace_counts['count'].sum() if not trace_counts.empty else 0
error_overall_count = error_counts['count'].sum() if not error_counts.empty else 0

trace_note = pd.DataFrame([{
    'file': 'trace.csv',
    'rows_for_components': int(trace_overall_count),
    'has_rows': 'yes' if trace_overall_count>0 else 'no'
}])

error_note = pd.DataFrame([{
    'file': 'error_logs.csv',
    'rows_for_components': int(error_overall_count),
    'has_rows': 'yes' if error_overall_count>0 else 'no'
}])

presence_summary = pd.concat([trace_note, error_note], ignore_index=True)

# Final compact outputs (as required, use variables rather than print)
metric_summary, log_summary, presence_summary

```
Out[2]:
```
import pandas as pd

# Use existing summary DataFrames from previous steps if available
try:
    metric_summary
    log_summary
    presence_summary
except NameError:
    # Fallback: read the previously computed summaries if not present (unlikely)
    metric_summary = pd.read_csv("metric_summary.csv") if False else pd.DataFrame()
    log_summary = pd.read_csv("log_summary.csv") if False else pd.DataFrame()
    presence_summary = pd.DataFrame()

components = ["carts", "catalogue", "orders", "payment", "user"]

def comp_metric_notes(df, comp):
    d = df[df['cmdb_id']==comp]
    if d.empty:
        return f"- {comp}: no metric KPIs found."
    total_kpis = d.shape[0]
    kpis_with_anom = (d['anomaly_count_in_window']>0).sum()
    total_anom_points = int(d['anomaly_count_in_window'].sum())
    # top KPI by anomaly_count, tie-breaker by total_points_in_window
    top = d.sort_values(['anomaly_count_in_window','total_points_in_window'], ascending=[False,False]).iloc[0]
    top_kpi = f"{top['kpi_name']} (anomalies={int(top['anomaly_count_in_window'])}, p95={float(top['p95_threshold']):g})"
    # earliest anomaly across KPIs
    times = d.loc[d['anomaly_count_in_window']>0,'earliest_anomaly_timestamp'].dropna().tolist()
    earliest = min(times) if times else None
    return (f"- {comp}: {total_kpis} metric KPIs monitored, {kpis_with_anom} KPI(s) had >=1 anomaly point "
            f"(total anomaly points={total_anom_points}). Top KPI: {top_kpi}. "
            f"Earliest metric anomaly: {earliest}.")

def comp_log_notes(df, comp):
    d = df[df['cmdb_id']==comp]
    if d.empty:
        return f"- {comp}: no log metrics found."
    total_logs = d.shape[0]
    logs_with_anom = (d['anomaly_count_in_window']>0).sum()
    total_anom_points = int(d['anomaly_count_in_window'].sum())
    top = d.sort_values(['anomaly_count_in_window','total_points_in_window'], ascending=[False,False]).iloc[0]
    top_log = f"{top['log_name']} (anomalies={int(top['anomaly_count_in_window'])}, p95={float(top['p95_threshold']):g})"
    times = d.loc[d['anomaly_count_in_window']>0,'earliest_anomaly_timestamp'].dropna().tolist()
    earliest = min(times) if times else None
    return (f"- {comp}: {total_logs} log metrics monitored, {logs_with_anom} log metric(s) had >=1 anomaly point "
            f"(total anomaly points={total_anom_points}). Top log metric: {top_log}. "
            f"Earliest log anomaly: {earliest}.")

# Build summary text
lines = []
lines.append("Summary for components: carts, catalogue, orders, payment, user (window 2024-01-18 16:27:59 to 2024-01-18 16:57:59 UTC)")
lines.append("")
lines.append("Metric results:")
for c in components:
    lines.append(comp_metric_notes(metric_summary, c))

lines.append("")
lines.append("Log results:")
for c in components:
    lines.append(comp_log_notes(log_summary, c))

# Presence of traces and error_logs for these components
trace_row = presence_summary[presence_summary['file']=='trace.csv']
error_row = presence_summary[presence_summary['file']=='error_logs.csv']
trace_has = trace_row['has_rows'].iloc[0] if not trace_row.empty else 'no'
trace_count = int(trace_row['rows_for_components'].iloc[0]) if not trace_row.empty else 0
error_has = error_row['has_rows'].iloc[0] if not error_row.empty else 'no'
error_count = int(error_row['rows_for_components'].iloc[0]) if not error_row.empty else 0

lines.append("")
lines.append(f"Trace data present for these components? {trace_has} (rows={trace_count})")
lines.append(f"Error log data present for these components? {error_has} (rows={error_count})")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp
0   catalogue      socket   6.000000e+00                      25                       25  2024-01-18T16:30:00+00:00
1      orders       error   0.000000e+00                      25                       25  2024-01-18T16:30:00+00:00
2     payment      socket   5.000000e+00                      25                       25  2024-01-18T16:30:00+00:00
3        user      socket   1.600000e+01                      25                       20  2024-01-18T16:30:00+00:00
4       carts      socket   1.400000e+01                      25                        3  2024-01-18T16:45:00+00:00
..        ...         ...            ...                     ...                      ...                        ...
26       user         cpu   8.526114e-01                      25                        2  2024-01-18T16:30:00+00:00
27       user  latency-50   3.405773e-03                      25                        2  2024-01-18T16:45:00+00:00
28       user  latency-90   7.099748e-03                      25                        2  2024-01-18T16:45:00+00:00
29       user         mem   8.165185e+06                      25                        2  2024-01-18T16:53:00+00:00
30       user    workload   1.935648e+01                      25                        2  2024-01-18T16:30:00+00:00

[31 rows x 6 columns],      cmdb_id         log_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp
0  catalogue  log.error_count           0.00                      25                       25  2024-01-18T16:30:00+00:00
1     orders  log.error_count           0.00                      25                       25  2024-01-18T16:30:00+00:00
2    payment  log.error_count           0.00                      25                       25  2024-01-18T16:30:00+00:00
3       user  log.error_count           0.00                      25                       25  2024-01-18T16:30:00+00:00
4      carts  log.error_count           0.00                      20                       20  2024-01-18T16:31:00+00:00
5  catalogue    log.row_count         167.60                      25                        2  2024-01-18T16:50:00+00:00
6     orders    log.row_count         131.60                      25                        2  2024-01-18T16:43:00+00:00
7    payment    log.row_count         105.80                      25                        2  2024-01-18T16:43:00+00:00
8       user    log.row_count         623.80                      25                        2  2024-01-18T16:43:00+00:00
9      carts    log.row_count         126.25                      20                        1  2024-01-18T16:51:00+00:00,              file  rows_for_components has_rows
0       trace.csv                    0       no
1  error_logs.csv                    0       no)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing loaded dataframes if present, otherwise load
try:
    df_metric
    df_log
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_log = pd.read_csv("log.csv")

# Ensure timestamps are parsed to UTC datetimes
for df in (df_metric, df_log):
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Parameters
components = ["catalogue","orders","payment","user","carts"]
window_start = pd.to_datetime("2024-01-18 16:27:59", utc=True)
window_end = pd.to_datetime("2024-01-18 16:57:59", utc=True)

# Helper to compute longest consecutive True run given an iterable of booleans
def longest_consecutive_true(seq):
    max_run = 0
    cur = 0
    for v in seq:
        if v:
            cur += 1
            if cur > max_run:
                max_run = cur
        else:
            cur = 0
    return max_run

# -------------------------
# 1) Metrics: kpi_name == 'socket'
# -------------------------
metric_socket_full = df_metric[
    (df_metric['kpi_name'] == 'socket') & (df_metric['cmdb_id'].isin(components))
].copy()

# Compute global P95 per component (full series)
p95_metric = metric_socket_full.groupby('cmdb_id', dropna=False)['value'].quantile(0.95).rename('p95_threshold').reset_index()
p95_metric = pd.DataFrame({'cmdb_id': components}).merge(p95_metric, on='cmdb_id', how='left')

# Filter to incident window
metric_socket_window = metric_socket_full[(metric_socket_full['timestamp'] >= window_start) & (metric_socket_full['timestamp'] <= window_end)].copy()

# Prepare summary rows per component
rows = []
for comp in components:
    p95 = p95_metric.loc[p95_metric['cmdb_id']==comp, 'p95_threshold'].iloc[0]
    comp_win = metric_socket_window[metric_socket_window['cmdb_id']==comp].sort_values('timestamp').copy()
    total_points = int(comp_win.shape[0])
    if pd.isna(p95) or total_points == 0:
        anomaly_count = 0
        earliest = None
        longest_run = 0
        samples_str = ""  # no anomalies to show
    else:
        comp_win['is_anom'] = comp_win['value'] >= p95
        anomaly_count = int(comp_win['is_anom'].sum())
        earliest = comp_win.loc[comp_win['is_anom'], 'timestamp'].min().isoformat() if anomaly_count>0 else None
        longest_run = longest_consecutive_true(comp_win['is_anom'].tolist())
        samples = comp_win.loc[comp_win['is_anom'], ['timestamp','value']].head(10)
        samples_list = [f"{ts.isoformat()}|{v}" for ts, v in zip(samples['timestamp'], samples['value'])]
        samples_str = "; ".join(samples_list)
    rows.append({
        'cmdb_id': comp,
        'kpi_name': 'socket',
        'p95_threshold': float(p95) if not pd.isna(p95) else float('nan'),
        'total_points_in_window': total_points,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest,
        'longest_consecutive_anomaly_run_length': int(longest_run),
        'anomaly_samples_up_to_10': samples_str
    })

metric_socket_summary = pd.DataFrame(rows)

# -------------------------
# 2) Logs: log_name == 'log.error_count'
# -------------------------
log_error_full = df_log[
    (df_log['log_name'] == 'log.error_count') & (df_log['cmdb_id'].isin(components))
].copy()

# Compute global P95 per component for logs (full series)
p95_log = log_error_full.groupby('cmdb_id', dropna=False)['value'].quantile(0.95).rename('p95_threshold').reset_index()
p95_log = pd.DataFrame({'cmdb_id': components}).merge(p95_log, on='cmdb_id', how='left')

# Filter to window
log_error_window = log_error_full[(log_error_full['timestamp'] >= window_start) & (log_error_full['timestamp'] <= window_end)].copy()

# Prepare summary rows per component
rows_l = []
for comp in components:
    p95 = p95_log.loc[p95_log['cmdb_id']==comp, 'p95_threshold'].iloc[0]
    comp_win = log_error_window[log_error_window['cmdb_id']==comp].sort_values('timestamp').copy()
    total_points = int(comp_win.shape[0])
    if total_points == 0:
        anomaly_count = 0
        earliest = None
        samples_str = ""
    else:
        # Even if p95 is 0 or NaN, still show window values; for anomaly counting require numeric p95
        if pd.isna(p95):
            comp_win['is_anom'] = False
            anomaly_count = 0
            earliest = None
        else:
            comp_win['is_anom'] = comp_win['value'] >= p95
            anomaly_count = int(comp_win['is_anom'].sum())
            earliest = comp_win.loc[comp_win['is_anom'], 'timestamp'].min().isoformat() if anomaly_count>0 else None
        samples = comp_win[['timestamp','value']].head(10)
        samples_list = [f"{ts.isoformat()}|{v}" for ts, v in zip(samples['timestamp'], samples['value'])]
        samples_str = "; ".join(samples_list)
    rows_l.append({
        'cmdb_id': comp,
        'log_name': 'log.error_count',
        'p95_threshold': float(p95) if not pd.isna(p95) else float('nan'),
        'total_points_in_window': total_points,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest,
        'window_samples_up_to_10': samples_str
    })

log_error_summary = pd.DataFrame(rows_l)

# Return compact summaries (variables only)
metric_socket_summary, log_error_summary

```
Out[3]:
```
Summary (components: catalogue, orders, payment, user, carts; window 2024-01-18 16:27:59–16:57:59 UTC)

Metric (kpi = "socket")
- catalogue: p95 = 6.00. 25 points in window, 25 anomalies (value >= p95). Earliest anomaly 2024-01-18T16:30:00+00:00. Longest consecutive anomaly run = 25 (persistent across the window). Samples show repeated values at/above 6.0.
- payment: p95 = 5.00. 25 points, 25 anomalies. Earliest 2024-01-18T16:30:00+00:00. Longest run = 25 (persistent).
- user: p95 = 16.00. 25 points, 20 anomalies. Earliest 2024-01-18T16:30:00+00:00. Longest run = 13 (long sustained anomaly period).
- orders: p95 = 19.88. 25 points, 2 anomalies. Earliest 2024-01-18T16:45:00+00:00. Longest run = 1 (brief, isolated spikes).
- carts: p95 = 14.00. 25 points, 3 anomalies. Earliest 2024-01-18T16:45:00+00:00. Longest run = 1 (brief spikes).

Logs (log_name = "log.error_count")
- catalogue: p95 = 0.0. 25 points in window, anomaly_count reported 25 (because value >= 0). Earliest reported = 2024-01-18T16:30:00+00:00. Window samples are zeros (no increase in error counts).
- orders: p95 = 0.0. 25 points, anomaly_count 25; earliest 2024-01-18T16:30:00+00:00. Window samples are zeros.
- payment: p95 = 0.0. 25 points, anomaly_count 25; earliest 2024-01-18T16:30:00+00:00. Window samples are zeros.
- user: p95 = 0.0. 25 points, anomaly_count 25; earliest 2024-01-18T16:30:00+00:00. Window samples are zeros.
- carts: p95 = 0.0. 20 points, anomaly_count 20; earliest 2024-01-18T16:31:00+00:00. Window samples are zeros.

Interpretation / key takeaways
- Strong, persistent socket-level anomalies during the incident window for catalogue and payment (entire window). The user service also shows sustained socket abnormalities; orders and carts show only brief spikes.
- log.error_count p95 = 0 leads to every point qualifying as ">= p95"; actual log samples in the window are zeros, so there is no evidence of increased error_count in logs despite socket anomalies.
- Overall, the dominant signal is socket metric problems (particularly catalogue and payment), while error logs do not show increased error counts in this window.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id kpi_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp  longest_consecutive_anomaly_run_length                           anomaly_samples_up_to_10
0  catalogue   socket           6.00                      25                       25  2024-01-18T16:30:00+00:00                                      25  2024-01-18T16:30:00+00:00|6.0; 2024-01-18T16:3...
1     orders   socket          19.88                      25                        2  2024-01-18T16:45:00+00:00                                       1  2024-01-18T16:45:00+00:00|20.016666666666666; ...
2    payment   socket           5.00                      25                       25  2024-01-18T16:30:00+00:00                                      25  2024-01-18T16:30:00+00:00|5.0; 2024-01-18T16:3...
3       user   socket          16.00                      25                       20  2024-01-18T16:30:00+00:00                                      13  2024-01-18T16:30:00+00:00|16.0; 2024-01-18T16:...
4      carts   socket          14.00                      25                        3  2024-01-18T16:45:00+00:00                                       1  2024-01-18T16:45:00+00:00|14.0; 2024-01-18T16:...,      cmdb_id         log_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp                            window_samples_up_to_10
0  catalogue  log.error_count            0.0                      25                       25  2024-01-18T16:30:00+00:00  2024-01-18T16:30:00+00:00|0; 2024-01-18T16:31:...
1     orders  log.error_count            0.0                      25                       25  2024-01-18T16:30:00+00:00  2024-01-18T16:30:00+00:00|0; 2024-01-18T16:31:...
2    payment  log.error_count            0.0                      25                       25  2024-01-18T16:30:00+00:00  2024-01-18T16:30:00+00:00|0; 2024-01-18T16:31:...
3       user  log.error_count            0.0                      25                       25  2024-01-18T16:30:00+00:00  2024-01-18T16:30:00+00:00|0; 2024-01-18T16:31:...
4      carts  log.error_count            0.0                      20                       20  2024-01-18T16:31:00+00:00  2024-01-18T16:31:00+00:00|0; 2024-01-18T16:32:...)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp parsed to UTC datetimes (rule 10)
if not pd.api.types.is_datetime64_any_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Parameters
components = ["catalogue","payment","user","orders","carts"]
window_start = pd.to_datetime("2024-01-18 16:27:59", utc=True)
window_end = pd.to_datetime("2024-01-18 16:57:59", utc=True)

# Helper to format timestamp or return None
def fmt_iso(ts):
    return None if pd.isna(ts) else pd.to_datetime(ts).isoformat()

# 1) KPI 'socket' - global P95 then window stats
metric_socket_full = df_metric[df_metric['kpi_name'] == 'socket'].copy()

# Compute global P95 per component (using full series, then align to requested components)
p95_socket = (
    metric_socket_full
    .groupby('cmdb_id', dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_threshold')
)

# Windowed records for socket
metric_socket_window = metric_socket_full[(metric_socket_full['timestamp'] >= window_start) & (metric_socket_full['timestamp'] <= window_end)].copy()

socket_rows = []
for comp in components:
    # global p95 for comp
    row = p95_socket[p95_socket['cmdb_id'] == comp]
    p95_val = float(row['p95_threshold'].iloc[0]) if not row.empty else float('nan')
    # window subset
    win = metric_socket_window[metric_socket_window['cmdb_id'] == comp].sort_values('timestamp')
    total_points = int(win.shape[0])
    if total_points == 0:
        max_val = np.nan
        mean_val = np.nan
        max_excess_pct = None
        earliest_anom = None
    else:
        max_val = float(win['value'].max())
        mean_val = float(win['value'].mean())
        if pd.isna(p95_val) or p95_val == 0:
            max_excess_pct = None
        else:
            max_excess_pct = float((max_val - p95_val) / p95_val * 100.0)
        # earliest anomaly timestamp where value >= p95 (only if p95 not NaN)
        if pd.isna(p95_val):
            earliest_anom = None
        else:
            anom_rows = win[win['value'] >= p95_val]
            earliest_anom = fmt_iso(anom_rows['timestamp'].min()) if not anom_rows.empty else None
    socket_rows.append({
        'cmdb_id': comp,
        'kpi_name': 'socket',
        'p95_threshold': (p95_val if not np.isnan(p95_val) else None),
        'total_points_in_window': total_points,
        'max_value_in_window': (None if pd.isna(max_val) else max_val),
        'mean_value_in_window': (None if pd.isna(mean_val) else mean_val),
        'max_excess_pct': max_excess_pct,
        'earliest_anomaly_timestamp': earliest_anom
    })

socket_summary = pd.DataFrame(socket_rows).head(5)

# 2) KPI 'workload' - global P15 then window anomalies (value <= p15)
metric_workload_full = df_metric[df_metric['kpi_name'] == 'workload'].copy()
p15_workload = (
    metric_workload_full
    .groupby('cmdb_id', dropna=False)['value']
    .quantile(0.15)
    .reset_index(name='p15_threshold')
)

metric_workload_window = metric_workload_full[(metric_workload_full['timestamp'] >= window_start) & (metric_workload_full['timestamp'] <= window_end)].copy()

workload_rows = []
for comp in components:
    row = p15_workload[p15_workload['cmdb_id'] == comp]
    if row.empty:
        # indicate missing workload for this component
        workload_rows.append({
            'cmdb_id': comp,
            'kpi_name': 'workload',
            'p15_threshold': None,
            'total_points_in_window': 0,
            'anomaly_count_in_window': 0,
            'earliest_anomaly_timestamp': None,
            'present': False
        })
        continue
    p15_val = float(row['p15_threshold'].iloc[0])
    win = metric_workload_window[metric_workload_window['cmdb_id'] == comp].sort_values('timestamp')
    total_points = int(win.shape[0])
    if total_points == 0:
        anomaly_count = 0
        earliest = None
    else:
        anoms = win[win['value'] <= p15_val]
        anomaly_count = int(anoms.shape[0])
        earliest = fmt_iso(anoms['timestamp'].min()) if not anoms.empty else None
    workload_rows.append({
        'cmdb_id': comp,
        'kpi_name': 'workload',
        'p15_threshold': p15_val,
        'total_points_in_window': total_points,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp': earliest,
        'present': True
    })

workload_summary = pd.DataFrame(workload_rows).head(5)

# 3) KPIs 'latency-50' and 'latency-90' - global P95 then window stats
lat_kpis = ['latency-50', 'latency-90']
lat_rows = []
for kpi in lat_kpis:
    full_kpi = df_metric[df_metric['kpi_name'] == kpi].copy()
    if full_kpi.empty:
        # no data for this kpi at all; still produce missing rows per component
        for comp in components:
            lat_rows.append({
                'cmdb_id': comp,
                'kpi_name': kpi,
                'p95_threshold': None,
                'total_points_in_window': 0,
                'anomaly_count_in_window': 0,
                'earliest_anomaly_timestamp': None
            })
        continue
    # compute global P95 per component for this KPI
    p95_df = full_kpi.groupby('cmdb_id', dropna=False)['value'].quantile(0.95).reset_index(name='p95_threshold')
    # windowed data for this KPI
    window_kpi = full_kpi[(full_kpi['timestamp'] >= window_start) & (full_kpi['timestamp'] <= window_end)].copy()
    for comp in components:
        row = p95_df[p95_df['cmdb_id'] == comp]
        if row.empty:
            p95_val = float('nan')
        else:
            p95_val = float(row['p95_threshold'].iloc[0])
        win = window_kpi[window_kpi['cmdb_id'] == comp].sort_values('timestamp')
        total_points = int(win.shape[0])
        if total_points == 0 or pd.isna(p95_val):
            anomaly_count = 0
            earliest = None
        else:
            anoms = win[win['value'] >= p95_val]
            anomaly_count = int(anoms.shape[0])
            earliest = fmt_iso(anoms['timestamp'].min()) if not anoms.empty else None
        lat_rows.append({
            'cmdb_id': comp,
            'kpi_name': kpi,
            'p95_threshold': (p95_val if not np.isnan(p95_val) else None),
            'total_points_in_window': total_points,
            'anomaly_count_in_window': anomaly_count,
            'earliest_anomaly_timestamp': earliest
        })

latency_summary = pd.DataFrame(lat_rows).head(10)  # up to 10 rows (5 components * 2 KPIs)

# Final compact outputs (variables only)
socket_summary, workload_summary, latency_summary

```
Out[4]:
```
summary = (
    "Summary (window 2024-01-18 16:27:59–16:57:59 UTC):\n\n"
    "1) KPI = 'socket' (global P95, window stats):\n"
    "- catalogue: p95=6.00, 25 points, max=6.00, mean=6.00, max_excess_pct=0.0%, earliest anomaly=2024-01-18T16:30:00+00:00 (persistent across window).\n"
    "- payment:   p95=5.00, 25 points, max=5.00, mean=5.00, max_excess_pct=0.0%, earliest anomaly=2024-01-18T16:30:00+00:00 (persistent).\n"
    "- user:      p95=16.00, 25 points, max=16.00, mean=15.834, max_excess_pct=0.0%, earliest anomaly=2024-01-18T16:30:00+00:00 (long sustained period).\n"
    "- orders:    p95=19.88, 25 points, max=20.20, mean=18.299, max_excess_pct≈1.61%, earliest anomaly=2024-01-18T16:45:00+00:00 (brief spikes).\n"
    "- carts:     p95=14.00, 25 points, max=14.15, mean=11.878, max_excess_pct≈1.07%, earliest anomaly=2024-01-18T16:45:00+00:00 (brief spikes).\n\n"
    "2) KPI = 'workload' (global P15, low-side anomalies value <= P15):\n"
    "- catalogue: p15≈3.9646, 25 points, 4 low-side anomalies, earliest 2024-01-18T16:32:00+00:00.\n"
    "- payment:   p15≈1.9554, 25 points, 4 low-side anomalies, earliest 2024-01-18T16:32:00+00:00.\n"
    "- user:      p15≈17.9721, 25 points, 4 low-side anomalies, earliest 2024-01-18T16:32:00+00:00.\n"
    "- orders:    p15≈1.9818, 25 points, 4 low-side anomalies, earliest 2024-01-18T16:32:00+00:00.\n"
    "- carts:     p15≈7.8467, 25 points, 4 low-side anomalies, earliest 2024-01-18T16:39:00+00:00.\n\n"
    "3) KPIs 'latency-50' and 'latency-90' (global P95, window anomalies):\n"
    "- latency-50:\n"
    "  * catalogue: p95≈0.003046, 25 pts, 2 anomalies, earliest 2024-01-18T16:45:00+00:00.\n"
    "  * payment:   p95≈0.002174, 25 pts, 2 anomalies, earliest 2024-01-18T16:53:00+00:00.\n"
    "  * user:      p95≈0.003406, 25 pts, 2 anomalies, earliest 2024-01-18T16:45:00+00:00.\n"
    "  * orders:    p95≈0.058146, 25 pts, 2 anomalies, earliest 2024-01-18T16:45:00+00:00.\n"
    "  * carts:     p95≈0.033889, 25 pts, 2 anomalies, earliest 2024-01-18T16:45:00+00:00.\n"
    "- latency-90:\n"
    "  * catalogue: p95≈0.004682, 25 pts, 2 anomalies, earliest 2024-01-18T16:45:00+00:00.\n"
    "  * payment:   p95≈0.004435, 25 pts, 2 anomalies, earliest 2024-01-18T16:53:00+00:00.\n"
    "  * user:      p95≈0.007100, 25 pts, 2 anomalies, earliest 2024-01-18T16:45:00+00:00.\n"
    "  * orders:    p95≈0.091784, 25 pts, 2 anomalies, earliest 2024-01-18T16:45:00+00:00.\n"
    "  * carts:     p95≈0.087022, 25 pts, 2 anomalies, earliest 2024-01-18T16:45:00+00:00.\n\n"
    "Key takeaways:\n"
    "- Socket metrics show persistent issues for catalogue and payment (values at or above P95 across the window) and sustained abnormality for user; orders and carts had only brief spikes above P95.\n"
    "- Workload shows occasional low-side anomalies (4 points each) across all components; these are drops below the P15 threshold.\n"
    "- Latency KPIs show brief spikes (2 points each), mostly around 16:45 UTC (payment had spikes near 16:53 UTC).\n"
    "Overall, the strongest signal is persistent socket-level abnormalities (catalogue/payment/user), while latency and workload show shorter/smaller anomalies."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id kpi_name  p95_threshold  total_points_in_window  max_value_in_window  mean_value_in_window  max_excess_pct earliest_anomaly_timestamp
0  catalogue   socket           6.00                      25                 6.00              6.000000        0.000000  2024-01-18T16:30:00+00:00
1    payment   socket           5.00                      25                 5.00              5.000000        0.000000  2024-01-18T16:30:00+00:00
2       user   socket          16.00                      25                16.00             15.834000        0.000000  2024-01-18T16:30:00+00:00
3     orders   socket          19.88                      25                20.20             18.299333        1.609658  2024-01-18T16:45:00+00:00
4      carts   socket          14.00                      25                14.15             11.878000        1.071429  2024-01-18T16:45:00+00:00,      cmdb_id  kpi_name  p15_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp  present
0  catalogue  workload       3.964553                      25                        4  2024-01-18T16:32:00+00:00     True
1    payment  workload       1.955350                      25                        4  2024-01-18T16:32:00+00:00     True
2       user  workload      17.972077                      25                        4  2024-01-18T16:32:00+00:00     True
3     orders  workload       1.981760                      25                        4  2024-01-18T16:32:00+00:00     True
4      carts  workload       7.846713                      25                        4  2024-01-18T16:39:00+00:00     True,      cmdb_id    kpi_name  p95_threshold  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp
0  catalogue  latency-50       0.003046                      25                        2  2024-01-18T16:45:00+00:00
1    payment  latency-50       0.002174                      25                        2  2024-01-18T16:53:00+00:00
2       user  latency-50       0.003406                      25                        2  2024-01-18T16:45:00+00:00
3     orders  latency-50       0.058146                      25                        2  2024-01-18T16:45:00+00:00
4      carts  latency-50       0.033889                      25                        2  2024-01-18T16:45:00+00:00
5  catalogue  latency-90       0.004682                      25                        2  2024-01-18T16:45:00+00:00
6    payment  latency-90       0.004435                      25                        2  2024-01-18T16:53:00+00:00
7       user  latency-90       0.007100                      25                        2  2024-01-18T16:45:00+00:00
8     orders  latency-90       0.091784                      25                        2  2024-01-18T16:45:00+00:00
9      carts  latency-90       0.087022                      25                        2  2024-01-18T16:45:00+00:00)```
```